# 07 — Model Evaluation & Comparison
Aggregate all model metrics, run the Diebold-Mariano test, and produce thesis-ready charts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import glob

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Load all metrics files

In [ ]:
import glob

metrics_files = glob.glob('../data/processed/metrics_*.csv')
if not metrics_files:
    print('No metrics files found yet — run notebooks 03, 04, 05 first.')
else:
    frames = []
    for f in metrics_files:
        df = pd.read_csv(f)
        frames.append(df)
    all_metrics = pd.concat(frames, ignore_index=True)

    # Fill optional columns so table is uniform
    for col in ['mape', 'wda']:
        if col not in all_metrics.columns:
            all_metrics[col] = np.nan

    all_metrics = all_metrics.sort_values('rmse').reset_index(drop=True)
    display(all_metrics)


## 2. Summary table (thesis-ready)

In [ ]:
metric_cols = [c for c in ['rmse', 'mae', 'mape', 'dir_acc', 'wda']
               if c in all_metrics.columns and all_metrics[c].notna().any()]

styled = all_metrics[['model'] + metric_cols].style

# Highlight best: lower is better for error metrics, higher for accuracy
lower_better = [c for c in ['rmse', 'mae', 'mape'] if c in metric_cols]
higher_better = [c for c in ['dir_acc', 'wda'] if c in metric_cols]

if lower_better:
    styled = styled.highlight_min(subset=lower_better, color='lightgreen')
if higher_better:
    styled = styled.highlight_max(subset=higher_better, color='lightgreen')

fmt = {c: '{:.6f}' for c in lower_better}
fmt.update({c: '{:.3f}' for c in higher_better})
if 'mape' in metric_cols:
    fmt['mape'] = '{:.2f}'
styled = styled.format(fmt)
display(styled)


## 3. Bar charts

In [ ]:
plot_metrics = [
    ('rmse',    'RMSE (lower = better)'),
    ('mae',     'MAE (lower = better)'),
    ('dir_acc', 'Directional Accuracy (higher = better)'),
    ('wda',     'Weighted DA (higher = better)'),
]
plot_metrics = [(m, lbl) for m, lbl in plot_metrics
                if m in all_metrics.columns and all_metrics[m].notna().any()]

n = len(plot_metrics)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]

palette = sns.color_palette('tab10', len(all_metrics))
for ax, (metric, label) in zip(axes, plot_metrics):
    ax.barh(all_metrics['model'], all_metrics[metric], color=palette)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel(metric.upper())
    if metric in ('dir_acc', 'wda'):
        ax.axvline(0.5, color='red', linestyle='--', lw=1, label='Random (0.5)')
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Diebold-Mariano test
Tests whether two models have statistically different forecast accuracy.
H₀: equal predictive accuracy.


In [ ]:
def diebold_mariano(e1: np.ndarray, e2: np.ndarray, h: int = 1) -> dict:
    """
    e1, e2 : forecast error arrays  (actual − predicted), same length
    h      : forecast horizon (1 for 1-step-ahead)
    Returns DM statistic and two-sided p-value.
    H₀: equal predictive accuracy.
    """
    d     = e1**2 - e2**2
    T     = len(d)
    d_bar = d.mean()

    # Newey-West long-run variance, h-1 autocovariance lags
    gamma0 = np.var(d, ddof=1)
    gammas = [np.cov(d[k:], d[:-k])[0, 1] for k in range(1, h)]
    nw_var = gamma0 + 2 * sum(gammas)

    dm_stat = d_bar / np.sqrt(max(nw_var, 1e-12) / T)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return {
        'DM stat':      round(dm_stat, 4),
        'p-value':      round(p_value, 4),
        'Reject H₀ (5%)': p_value < 0.05,
    }


# ── Load predictions saved by each notebook ──────────────────────────────────
# Each model notebook should save a CSV with columns: actual, predicted
# e.g.  pd.DataFrame({'actual': actuals, 'predicted': preds}).to_csv(...)
# We attempt to load them here; run the DM test for every pair found.

import itertools

pred_files = {
    'ARIMA rolling':   '../data/processed/preds_arima_rol.csv',
    'ARIMAX rolling':  '../data/processed/preds_arimax_rol.csv',
    'U-MIDAS':         '../data/processed/preds_umidas.csv',
    'LSTM':            '../data/processed/preds_lstm.csv',
}

loaded = {}
for name, fpath in pred_files.items():
    try:
        df = pd.read_csv(fpath)
        loaded[name] = df
        print(f'Loaded {name}: {df.shape}')
    except FileNotFoundError:
        print(f'Not found yet: {fpath}')

if len(loaded) >= 2:
    pairs = list(itertools.combinations(loaded.keys(), 2))
    rows  = []
    for m1, m2 in pairs:
        d1, d2 = loaded[m1], loaded[m2]
        # align on common index
        merged = d1.rename(columns={'predicted': 'p1'}).join(
                     d2.rename(columns={'predicted': 'p2'})['p2'], how='inner')
        merged = merged.join(d1['actual'], how='inner', rsuffix='_dup')
        actuals_ = merged['actual'].values
        e1 = actuals_ - merged['p1'].values
        e2 = actuals_ - merged['p2'].values
        result = diebold_mariano(e1, e2)
        rows.append({'Model 1': m1, 'Model 2': m2, **result})
    dm_df = pd.DataFrame(rows)
    display(dm_df)
else:
    print('Need at least 2 prediction files to run DM tests.')
    print('Each model notebook should save:')
    print('  pd.DataFrame({"actual": actuals, "predicted": preds})')
    print('  .to_csv("../data/processed/preds_<model>.csv", index=False)')


## 5. Sentiment ablation — does it help?

In [ ]:
# Compare with-sentiment vs without-sentiment models
# Group models by whether they include sentiment
if 'with_sentiment' in all_metrics.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    for has_sent, grp in all_metrics.groupby('with_sentiment'):
        label = 'With sentiment' if has_sent else 'Without sentiment'
        ax.scatter(grp['model'], grp['rmse'], label=label, s=80)
    ax.set_title('RMSE: With vs Without Sentiment')
    ax.set_ylabel('RMSE')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Add a with_sentiment column to your metrics CSVs to enable this chart.')

## 6. Silver Squeeze episode zoom-in
Key test of the retail-sentiment hypothesis: do sentiment models outperform during Feb 2021?


In [ ]:
prices = pd.read_csv('../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
prices.index = prices.index.tz_localize(None)

squeeze = prices['silver']['2021-01-01':'2021-04-30']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(squeeze, lw=1.5, color='steelblue')
ax.axvspan('2021-01-28', '2021-02-05', alpha=0.2, color='red',
           label='Silver Squeeze (Jan 28 – Feb 5)')
ax.set_title('Silver Price — Squeeze Episode')
ax.set_ylabel('USD/oz')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Export final metrics table (LaTeX)

In [ ]:
latex = all_metrics.to_latex(index=False, float_format='%.6f',
                              caption='Out-of-sample forecast accuracy by model',
                              label='tab:results')
print(latex)

with open('../data/processed/results_table.tex', 'w') as f:
    f.write(latex)
print('Saved LaTeX table.')